In [5]:
import jax
import jax.numpy as jnp


def h(x):
    return jnp.concatenate([x, x**2, jnp.array([x[0] * x[1]])])


x = jnp.array([1.0, 2.0, 3.0])
v = jnp.array([0.1, 0.2, 0.3])

# Compute grad f
jac_f = jax.jacobian(h)

# Compute Hessian-vector product using jax.jvp on grad_f
hvp_fast = jax.jvp(jac_f, (x,), (v,))[1]

hessian = jax.hessian(h)(x)
hvp_exact = hessian @ v

print("Value:", h(x))
print("Jacobian", jac_f(x), jac_f(x).shape)
print("Hessian", hessian.shape)
print("hvp fast", hvp_fast, hvp_fast.shape)
print("hvp exact", hvp_exact, hvp_exact.shape)


Value: [1. 2. 3. 1. 4. 9. 2.]
Jacobian [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]
 [2. 0. 0.]
 [0. 4. 0.]
 [0. 0. 6.]
 [2. 1. 0.]] (7, 3)
Hessian (7, 3, 3)
hvp fast [[0.  0.  0. ]
 [0.  0.  0. ]
 [0.  0.  0. ]
 [0.2 0.  0. ]
 [0.  0.4 0. ]
 [0.  0.  0.6]
 [0.2 0.1 0. ]] (7, 3)
hvp exact [[0.  0.  0. ]
 [0.  0.  0. ]
 [0.  0.  0. ]
 [0.2 0.  0. ]
 [0.  0.4 0. ]
 [0.  0.  0.6]
 [0.2 0.1 0. ]] (7, 3)


In [ ]:
a = jnp.array([1, 2])
b = jnp.array([3, 4])

a * b

In [2]:
# f = y.T @ h(x)

# Inputs
x = jnp.array([0.1, 0.2, 0.3])
y = jnp.array([1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0])
v = jnp.array([0.15, 0.25, 0.35])


# VJP of h(x) w.r.t. x, pulled back by y
def vjp_h_y(x):
    _, vjp_fun = jax.vjp(h, x)
    return vjp_fun(y)[0]


# Linearize it once at x
primals_out, pullback = jax.linearize(vjp_h_y, x)

# Now HVP is just pullback applied to v
hvp = pullback(v)

print("fast hvp:", hvp)


def f(x):
    return jnp.dot(y, h(x))


H = jax.hessian(f)(x)
print("exact hvp:", H @ v)
print("hessian:", H)
jnp.linalg.eigvals(H)

fast hvp: [2.95      3.5500002 4.2      ]
exact hvp: [2.95      3.5500002 4.2      ]
hessian: [[ 8.  7.  0.]
 [ 7. 10.  0.]
 [ 0.  0. 12.]]


Array([ 1.9289322+0.j, 16.071068 +0.j, 12.       +0.j], dtype=complex64)

In [3]:
hx, pullback_h = jax.linearize(h, x)


# Gauss Newton Hessian Apprximation
# Define the Gram HVP with y scaling
def gram_hvp_with_y(v):
    _, u = jax.jvp(h, (x,), (v,))
    _, pullback_h = jax.vjp(h, x)
    u_scaled = jnp.diag(y**2) @ u
    return pullback_h(u_scaled)[0]


# Test for a vector v
hvp_result = gram_hvp_with_y(v)
print("Gram HVP with y:", hvp_result)


def f(x):
    return jnp.diag(y) @ h(x)


print(jax.jacobian(f)(x).T @ jax.jacobian(f)(x))

Gram HVP with y: [0.7850001 2.2695    7.686    ]
[[ 3.6         0.97999996  0.        ]
 [ 0.97999996  8.49        0.        ]
 [ 0.          0.         21.960001  ]]


In [4]:
from typing import Callable

import jax
import jax.numpy as jnp
from flax import struct
from jax import flatten_util


@struct.dataclass
class Container:
    a: float
    x: jax.Array
    y: jax.Array


conainer = Container(a=5.5, x=jnp.array([1.0, 2.0]), y=jnp.array([[3.0, 4.0], [5.0, 6.0]]))

# Flatten into a 1D vector
flat_params, unravel_fn = flatten_util.ravel_pytree(conainer)

print(flat_params)
# [1. 2. 3. 4. 5. 6.]

# You can recover the original structure:
original_params = unravel_fn(flat_params)
print(original_params)
print(isinstance(unravel_fn, Callable))
print(isinstance(original_params, Container))

AttributeError: module 'jax' has no attribute 'flatten_util'